In [18]:
import os
import numpy as np
import pydicom
from PIL import Image
from tqdm import tqdm
from skimage.transform import resize

In [19]:
# === CONFIG ===
dcm_dir = r"raw_data55k"
png_dir = r"raw_png"
TARGET_SIZE = 256

os.makedirs(png_dir, exist_ok=True)

In [20]:
def safe_id(dcm, dcm_path):
    uid = str(getattr(dcm, "SOPInstanceUID", "")).strip()
    # UID valid biasanya digit dan titik; dataset lo banyak "ID_..."
    if uid and all(c.isdigit() or c == "." for c in uid):
        return uid
    return os.path.splitext(os.path.basename(dcm_path))[0]

def get_metadata(dcm):
    intercept = float(getattr(dcm, "RescaleIntercept", 0.0))
    slope = float(getattr(dcm, "RescaleSlope", 1.0))
    return intercept, slope

def window_wlww_to_01(img, wc, ww, intercept, slope, invert=False):
    hu = img.astype(np.float32) * slope + intercept
    low  = wc - ww / 2.0
    high = wc + ww / 2.0
    hu = np.clip(hu, low, high)
    out = (hu - low) / (high - low)  # 0..1
    if invert:
        out = 1.0 - out
    return out.astype(np.float32)

def prepare_image(dcm_path, out_dir, target_size=256):
    try:
        dcm = pydicom.dcmread(dcm_path, force=True)
        img = dcm.pixel_array

        # multiframe -> ambil frame pertama
        if img.ndim == 3:
            img = img[0]

        # pastikan 2D
        if img.ndim != 2:
            raise ValueError(f"Unexpected pixel_array shape: {img.shape}")

        intercept, slope = get_metadata(dcm)
        invert = (getattr(dcm, "PhotometricInterpretation", "") == "MONOCHROME1")

        # Windows (WL, WW)
        # RGB order: [blood, brain, bone]
        blood = window_wlww_to_01(img, 75, 215, intercept, slope, invert=invert)
        brain = window_wlww_to_01(img, 40, 80,  intercept, slope, invert=invert)
        bone  = window_wlww_to_01(img, 600, 2800, intercept, slope, invert=invert)

        rgb = np.stack([blood, brain, bone], axis=-1)  # float32 0..1

        # resize -> float32
        rgb_resized = resize(
            rgb,
            (target_size, target_size, 3),
            preserve_range=True,
            anti_aliasing=True
        ).astype(np.float32)

        # === normalisasi akhir (aman) ===
        rgb_norm = np.clip(rgb_resized, 0.0, 1.0).astype(np.float32)

        # save PNG uint8
        rgb8 = (rgb_norm * 255.0).round().astype(np.uint8)
        rgb8 = np.ascontiguousarray(rgb8)

        img_id = safe_id(dcm, dcm_path)
        out_path = os.path.join(out_dir, f"{img_id}.png")

        Image.fromarray(rgb8).save(out_path)
        return True

    except Exception as e:
        print(f"Error processing {dcm_path}: {e}")
        return False


In [ ]:
# === Run conversion ===
dcm_files = [os.path.join(dcm_dir, f) for f in os.listdir(dcm_dir) if f.lower().endswith(".dcm")]

ok, fail = 0, 0
for dcm_path in tqdm(dcm_files, desc="Converting DICOM to PNG"):
    if prepare_image(dcm_path, png_dir, TARGET_SIZE):
        ok += 1
    else:
        fail += 1

print(f"\nDone. Success: {ok} | Fail: {fail}")

In [2]:
import os
import pydicom
import numpy as np
import cv2
from tqdm import tqdm

In [13]:
# === Fungsi pendukung ===

def get_id(dcm):
    """Ambil SOPInstanceUID dari DICOM sebagai nama unik."""
    return dcm.SOPInstanceUID

def get_metadata(dcm):
    """Ambil metadata window dan konversi jika perlu."""
    intercept = getattr(dcm, "RescaleIntercept", 0)
    slope = getattr(dcm, "RescaleSlope", 1)
    return intercept, slope

def window_image(img, window_center, window_width, intercept, slope):
    """Terapkan windowing ke citra CT sesuai center dan width."""
    img = img * slope + intercept
    img_min = window_center - window_width / 2
    img_max = window_center + window_width / 2
    img = np.clip(img, img_min, img_max)
    return img

def normalize_minmax(img):
    """Normalisasi citra ke rentang 0–1."""
    img = img - np.min(img)
    if np.ptp(img) == 0:
        return img
    img = img / np.ptp(img)
    return img

def prepare_image(dcm_path, png_path):
    """Konversi 1 file DICOM menjadi PNG 3 channel (brain, blood, bone)."""
    try:
        dcm = pydicom.dcmread(dcm_path)
        img = dcm.pixel_array.astype(np.float32)
        intercept, slope = get_metadata(dcm)

        # 3 jendela berbeda
        windows = {
            "brain":  (40, 80),
            "blood":  (80, 200),
            "bone":   (600, 2000)
        }

        processed = []
        for name, (center, width) in windows.items():
            win_img = window_image(img, center, width, intercept, slope)
            win_img = normalize_minmax(win_img)
            processed.append(win_img)

        # Satukan jadi 3 channel (RGB)
        img_rgb = np.stack(processed, axis=-1)
        img_rgb = (img_rgb * 255).astype(np.uint8)

        img_id = get_id(dcm)
        out_path = os.path.join(png_path, f"{img_id}.png")
        cv2.imwrite(out_path, img_rgb)

    except Exception as e:
        print(f"Error processing {dcm_path}: {e}")


In [14]:
# === Jalankan konversi ===

dcm_dir = r"raw_data55k"
png_dir = r"raw_png"
os.makedirs(png_dir, exist_ok=True)

In [ ]:
dcm_files = [os.path.join(dcm_dir, f) for f in os.listdir(dcm_dir) if f.endswith(".dcm")]

for dcm_path in tqdm(dcm_files, desc="Converting DICOM to PNG"):
    prepare_image(dcm_path, png_dir)